In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from tqdm.notebook import tqdm

from app.data_loader import DataLoader
from app.prompts import get_template
from app.concepts_extraction import extract_key_concepts
from app.schemas import ModelEvaluation
from app.grading import grade_answer
from app.utils import write_file_with_uuid

In [ ]:
test_cases = DataLoader("../data/generated_questions").test_cases

In [ ]:
test_cases = {
    k: test_cases[k]
    for k in test_cases
    if k
    in [
        "08e8b29c583d4d3ca59749bcfb29fc7c",
        "0c5d8358c3c246958e6a1ccd80278b02",
    ]
}

In [ ]:
configs = {
    "tlite-8b": {"full_name": "t-tech/T-lite-it-2.1:Q4_K_M", "temperature": 1},
    "vikhr-8b": {"full_name": "rscr/vikhr_llama3.1_8b:Q4_K_M", "temperature": 1},
    "llama3-8b": {"full_name": "llama3:8b-instruct-q4_K_M", "temperature": 0.9},
}

In [ ]:
key_concepts = []

system_concept_template = get_template("concepts_extraction/system")
user_concept_template = get_template("concepts_extraction/user")

cases_pbar = tqdm(test_cases, "Извлечение ключевых понятий", len(test_cases), leave=True)

for case_uuid in cases_pbar:
    case = test_cases[case_uuid]
    
    concepts, _ = extract_key_concepts(
        "qwen2.5:3b",
        0.9,
        system_concept_template.render(),
        user_concept_template.render({"question": case.question}),
    )
    key_concepts.append(concepts)

In [ ]:
system_grading_template = get_template("grading/system")

In [ ]:
configs_pbar = tqdm(configs, "Модель", len(configs), leave=True, position=0)

for config_name in configs_pbar:
    config = configs[config_name]

    model = config["full_name"]
    temperature = config["temperature"]

    case_pbar = tqdm(enumerate(test_cases), "Вопросы", len(test_cases), leave=False, position=1)

    for case_idx, test_case_uuid in case_pbar:
        test_case = test_cases[test_case_uuid]

        concepts = key_concepts[case_idx]

        answer_pbar = tqdm(
            enumerate(test_case.answers),
            total=len(test_case.answers),
            desc=f"Ответы для вопроса {case_idx+1}",
            position=2,
            leave=False,
        )

        for answer_idx, answer in answer_pbar:
            try:
                system_prompt = system_grading_template.render(
                    {
                        "question": test_case.question,
                        "key_concepts": concepts.concepts,
                    }
                )
                grading_result, response = grade_answer(
                    model,
                    temperature,
                    system_prompt,
                    answer.text,
                    len(concepts.concepts),
                )
            except Exception as e:
                print(e)
                continue

            evaluation = ModelEvaluation(
                case_uuid=test_case_uuid,
                model=config_name,
                temperature=temperature,
                question=test_case.question,
                key_concepts=concepts,
                answer=answer,
                grading_results=grading_result,
                eval_duration=response.eval_duration,
                eval_count=response.eval_count,
            )

            write_file_with_uuid(
                f"../output/full_grading/{config_name}-{temperature}",
                evaluation.model_dump_json(indent=2),
            )